# JAX adata EDA

Basic exploratory analysis for the downloaded JAX AnnData files: file inventory, metadata summaries, embryonic staging exploration, and a full-data UMAP.

In [ ]:
from pathlib import Path
import gc
import os
import re
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import umap
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("ADATA_DIR", PROJECT_ROOT / "downloads" / "adata")).resolve()
OUT_DIR = Path(os.environ.get("EDA_OUT_DIR", PROJECT_ROOT / "outputs" / "jax_adata_eda")).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = [
    "adata_JAX_dataset_1.h5ad",
    "adata_JAX_dataset_2.h5ad",
    "adata_JAX_dataset_3.h5ad",
    "adata_JAX_dataset_4.h5ad",
    "df_cell.csv",
    "df_gene.csv",
]
H5AD_FILES = [DATA_DIR / f"adata_JAX_dataset_{i}.h5ad" for i in range(1, 5)]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUT_DIR}")

## File Inventory

In [ ]:
inventory = []
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    inventory.append({
        "file": name,
        "exists": path.exists(),
        "size_gb": path.stat().st_size / 1024**3 if path.exists() else np.nan,
    })
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT_DIR / "file_inventory.csv", index=False)
display(inventory)

missing = inventory.loc[~inventory["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError(f"Missing expected files: {missing}")

## AnnData Structure and Metadata

In [ ]:
def close_backed(adata):
    if getattr(adata, "isbacked", False) and getattr(adata, "file", None) is not None:
        adata.file.close()

summaries = []
obs_frames = []
obsm_keys_by_file = {}
var_names_by_file = {}

for path in H5AD_FILES:
    a = sc.read_h5ad(path, backed="r")
    summaries.append({
        "file": path.name,
        "n_obs": a.n_obs,
        "n_vars": a.n_vars,
        "n_obs_columns": len(a.obs.columns),
        "n_var_columns": len(a.var.columns),
        "obsm_keys": ",".join(a.obsm.keys()),
        "layers": ",".join(a.layers.keys()),
    })
    obs = a.obs.copy()
    obs["source_file"] = path.name
    obs_frames.append(obs)
    obsm_keys_by_file[path.name] = list(a.obsm.keys())
    var_names_by_file[path.name] = pd.Index(a.var_names.astype(str))
    close_backed(a)

adata_summary = pd.DataFrame(summaries)
obs_all = pd.concat(obs_frames, axis=0, join="outer", copy=False)
obs_all.to_csv(OUT_DIR / "obs_metadata_all.csv")
adata_summary.to_csv(OUT_DIR / "adata_summary.csv", index=False)

display(adata_summary)
print(f"Combined metadata rows: {obs_all.shape[0]:,}")
print(f"Combined metadata columns: {obs_all.shape[1]:,}")

In [ ]:
common_genes = None
for genes in var_names_by_file.values():
    common_genes = genes if common_genes is None else common_genes.intersection(genes)

gene_overlap = pd.DataFrame({
    "metric": ["common_genes_across_h5ad"],
    "value": [len(common_genes)],
})
gene_overlap.to_csv(OUT_DIR / "gene_overlap.csv", index=False)
display(gene_overlap)

In [ ]:
def example_values(s):
    vals = s.dropna().astype(str).unique()[:5]
    return "; ".join(vals)

metadata_summary = pd.DataFrame({
    "column": obs_all.columns,
    "dtype": [str(obs_all[c].dtype) for c in obs_all.columns],
    "missing_fraction": [float(obs_all[c].isna().mean()) for c in obs_all.columns],
    "n_unique": [int(obs_all[c].nunique(dropna=True)) for c in obs_all.columns],
    "examples": [example_values(obs_all[c]) for c in obs_all.columns],
})
metadata_summary.sort_values(["missing_fraction", "n_unique"], ascending=[True, False]).to_csv(
    OUT_DIR / "metadata_column_summary.csv", index=False
)
display(metadata_summary.head(30))

## Metadata Plots

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR / name, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=adata_summary, x="file", y="n_obs", color="#4c78a8")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Cells")
plt.xlabel("")
plt.title("Cells per AnnData file")
savefig("cells_per_file.png")

completeness = metadata_summary.sort_values("missing_fraction").head(40).copy()
completeness["present_fraction"] = 1 - completeness["missing_fraction"]
plt.figure(figsize=(8, max(4, 0.18 * len(completeness))))
sns.barplot(data=completeness, y="column", x="present_fraction", color="#59a14f")
plt.xlabel("Present fraction")
plt.ylabel("")
plt.title("Most complete metadata columns")
savefig("metadata_completeness_top40.png")

## Embryonic Staging and Time Metadata

In [ ]:
stage_regex = re.compile(r"stage|embry|gest|age|time|dpc|\be\d|day", re.IGNORECASE)
stage_cols = [c for c in obs_all.columns if stage_regex.search(str(c))]

stage_summary = metadata_summary[metadata_summary["column"].isin(stage_cols)].sort_values("column")
stage_summary.to_csv(OUT_DIR / "staging_metadata_summary.csv", index=False)
display(stage_summary)

def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")[:80]

for col in stage_cols[:12]:
    s = obs_all[col]
    if pd.api.types.is_numeric_dtype(s):
        plt.figure(figsize=(7, 4))
        sns.histplot(s.dropna(), bins=50, color="#f28e2b")
        plt.title(col)
        plt.xlabel(col)
        savefig(f"stage_hist_{safe_name(col)}.png")
    else:
        counts = s.astype(str).replace("nan", np.nan).dropna().value_counts().head(40).reset_index()
        counts.columns = [col, "n_cells"]
        counts.to_csv(OUT_DIR / f"stage_counts_{safe_name(col)}.csv", index=False)
        plt.figure(figsize=(8, max(4, 0.22 * len(counts))))
        sns.barplot(data=counts, y=col, x="n_cells", color="#f28e2b")
        plt.title(col)
        plt.xlabel("Cells")
        plt.ylabel("")
        savefig(f"stage_counts_{safe_name(col)}.png")

## Full-Data UMAP

In [ ]:
STREAM_CHUNK_CELLS = int(os.environ.get("STREAM_CHUNK_CELLS", "5000"))
N_HVG = int(os.environ.get("N_HVG", "2000"))
N_PCS = int(os.environ.get("N_PCS", "50"))
UMAP_TRAIN_CELLS = int(os.environ.get("UMAP_TRAIN_CELLS", "250000"))
UMAP_TRANSFORM_BATCH = int(os.environ.get("UMAP_TRANSFORM_BATCH", "50000"))
UMAP_PLOT_CELLS = int(os.environ.get("UMAP_PLOT_CELLS", "500000"))
RANDOM_SEED = int(os.environ.get("RANDOM_SEED", "7"))
NORMALIZE_LOG1P = os.environ.get("NORMALIZE_LOG1P", "auto")

progress_log = OUT_DIR / "streaming_umap_progress.log"
def log_progress(message):
    print(message, flush=True)
    with progress_log.open("a") as handle:
        handle.write(message + "\n")

total_cells = int(adata_summary["n_obs"].sum())
n_vars = int(adata_summary["n_vars"].iloc[0])
rng = np.random.default_rng(RANDOM_SEED)
log_progress(f"Streaming full-data UMAP for {total_cells:,} cells x {n_vars:,} genes")

def normalize_log1p_chunk(x):
    if sparse.issparse(x):
        x = x.astype(np.float32).tocsr(copy=True)
        counts = np.asarray(x.sum(axis=1)).ravel().astype(np.float32)
        scale = np.divide(1e4, counts, out=np.zeros_like(counts), where=counts > 0)
        x = x.multiply(scale[:, None]).tocsr()
        x.data = np.log1p(x.data)
        return x
    x = np.asarray(x, dtype=np.float32)
    counts = x.sum(axis=1).astype(np.float32)
    scale = np.divide(1e4, counts, out=np.zeros_like(counts), where=counts > 0)
    x = x * scale[:, None]
    return np.log1p(x, out=x)

def chunk_looks_like_counts(x):
    vals = x.data if sparse.issparse(x) else np.asarray(x).ravel()
    vals = vals[: min(100_000, vals.shape[0])]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return False
    return np.nanmax(vals) > 50 and np.allclose(vals[: min(10_000, vals.size)], np.round(vals[: min(10_000, vals.size)]))

def maybe_normalize(x, do_normalize):
    return normalize_log1p_chunk(x) if do_normalize else (x.astype(np.float32) if sparse.issparse(x) else np.asarray(x, dtype=np.float32))

def iter_matrix_chunks(paths, var_idx=None, chunk_size=STREAM_CHUNK_CELLS):
    global_offset = 0
    for path in paths:
        a = sc.read_h5ad(path, backed="r")
        for start in range(0, a.n_obs, chunk_size):
            end = min(start + chunk_size, a.n_obs)
            x = a.X[start:end, :]
            if var_idx is not None:
                x = x[:, var_idx]
            yield path.name, global_offset + start, global_offset + end, x
        global_offset += a.n_obs
        close_backed(a)

first_chunk = next(iter_matrix_chunks(H5AD_FILES, chunk_size=min(STREAM_CHUNK_CELLS, 5000)))[3]
do_normalize = chunk_looks_like_counts(first_chunk) if NORMALIZE_LOG1P == "auto" else NORMALIZE_LOG1P == "1"
log_progress(f"Normalize/log1p chunks: {do_normalize}")
del first_chunk

hvg_path = OUT_DIR / "streaming_hvg_genes.csv"
if hvg_path.exists() and os.environ.get("REUSE_STREAMING_HVG", "1") == "1":
    hvg_genes = pd.read_csv(hvg_path)["gene"].astype(str)
    hvg_idx = common_genes.get_indexer(hvg_genes)
    hvg_idx = hvg_idx[hvg_idx >= 0]
    log_progress(f"Reusing {len(hvg_idx):,} streaming HVGs from {hvg_path}")
else:
    sum_x = np.zeros(n_vars, dtype=np.float64)
    sum_x2 = np.zeros(n_vars, dtype=np.float64)
    seen = 0
    for file_name, start, end, x in iter_matrix_chunks(H5AD_FILES):
        x = maybe_normalize(x, do_normalize)
        sum_x += np.asarray(x.sum(axis=0)).ravel()
        sum_x2 += np.asarray(x.power(2).sum(axis=0)).ravel() if sparse.issparse(x) else np.square(x).sum(axis=0)
        seen += end - start
        if seen % 500000 == 0 or seen == total_cells:
            log_progress(f"HVG pass: {seen:,}/{total_cells:,} cells")
        del x
    mean_x = sum_x / seen
    var_x = np.maximum(sum_x2 / seen - mean_x**2, 0)
    hvg_idx = np.argsort(var_x)[-min(N_HVG, n_vars):]
    hvg_idx.sort()
    pd.DataFrame({"gene": common_genes[hvg_idx].astype(str), "variance": var_x[hvg_idx]}).to_csv(hvg_path, index=False)
    log_progress(f"Selected {len(hvg_idx):,} streaming HVGs")

n_pcs = min(N_PCS, len(hvg_idx) - 1)
pca_train_n = min(int(os.environ.get("PCA_TRAIN_CELLS", "250000")), total_cells)
pca_train_idx = np.sort(rng.choice(total_cells, size=pca_train_n, replace=False))
np.save(OUT_DIR / "pca_train_indices.npy", pca_train_idx)
sample_blocks = []
cursor = 0
for file_name, start, end, x in iter_matrix_chunks(H5AD_FILES, var_idx=hvg_idx):
    take = pca_train_idx[(pca_train_idx >= start) & (pca_train_idx < end)] - start
    if take.size:
        x = maybe_normalize(x, do_normalize)
        block = x[take, :] if sparse.issparse(x) else np.asarray(x, dtype=np.float32)[take, :]
        sample_blocks.append(block.tocsr() if sparse.issparse(block) else sparse.csr_matrix(block))
        cursor += take.size
        log_progress(f"PCA/SVD sample collection: {cursor:,}/{pca_train_n:,} cells")
    del x
sample_matrix = sparse.vstack(sample_blocks, format="csr").astype(np.float32)
del sample_blocks
log_progress(f"Fitting TruncatedSVD on {sample_matrix.shape[0]:,} sampled cells x {sample_matrix.shape[1]:,} HVGs")
svd = TruncatedSVD(n_components=n_pcs, random_state=RANDOM_SEED)
svd.fit(sample_matrix)
del sample_matrix

pca_path = OUT_DIR / "full_streaming_pca.npy"
pca_scores = np.lib.format.open_memmap(pca_path, mode="w+", dtype=np.float32, shape=(total_cells, n_pcs))
for file_name, start, end, x in iter_matrix_chunks(H5AD_FILES, var_idx=hvg_idx):
    x = maybe_normalize(x, do_normalize)
    pca_scores[start:end, :] = svd.transform(x).astype(np.float32)
    if end % 500000 == 0 or end == total_cells:
        log_progress(f"SVD transform pass: {end:,}/{total_cells:,} cells")
    del x
pca_scores.flush()

pd.DataFrame({
    "pc": np.arange(1, n_pcs + 1),
    "explained_variance_ratio": svd.explained_variance_ratio_,
}).to_csv(OUT_DIR / "streaming_pca_variance.csv", index=False)

train_n = min(UMAP_TRAIN_CELLS, total_cells)
train_idx = np.sort(rng.choice(total_cells, size=train_n, replace=False))
np.save(OUT_DIR / "umap_train_indices.npy", train_idx)
reducer = umap.UMAP(
    n_neighbors=int(os.environ.get("N_NEIGHBORS", "15")),
    min_dist=float(os.environ.get("UMAP_MIN_DIST", "0.5")),
    metric="euclidean",
    random_state=RANDOM_SEED,
    low_memory=True,
    verbose=True,
)
log_progress(f"Fitting UMAP on {train_n:,} sampled cells, then projecting all cells.")
reducer.fit(np.asarray(pca_scores[train_idx, :]))

coords_path = OUT_DIR / "full_umap_streaming.npy"
coords = np.lib.format.open_memmap(coords_path, mode="w+", dtype=np.float32, shape=(total_cells, 2))
for start in range(0, total_cells, UMAP_TRANSFORM_BATCH):
    end = min(start + UMAP_TRANSFORM_BATCH, total_cells)
    coords[start:end, :] = reducer.transform(np.asarray(pca_scores[start:end, :])).astype(np.float32)
    if end % 500000 == 0 or end == total_cells:
        log_progress(f"UMAP transform: {end:,}/{total_cells:,} cells")
coords.flush()

pd.DataFrame(np.asarray(coords), columns=["UMAP1", "UMAP2"]).to_csv(
    OUT_DIR / "full_umap_streaming_coordinates.csv.gz", index=False
)

plt.figure(figsize=(7, 6))
plt.hexbin(coords[:, 0], coords[:, 1], gridsize=450, bins="log", mincnt=1, cmap="viridis")
plt.colorbar(label="log10(cell count)")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Full-data streaming UMAP density")
savefig("full_umap_streaming_density.png")

plot_n = min(UMAP_PLOT_CELLS, total_cells)
plot_idx = np.sort(rng.choice(total_cells, size=plot_n, replace=False))
umap_df = pd.DataFrame(np.asarray(coords[plot_idx, :]), columns=["UMAP1", "UMAP2"])
umap_df["source_file"] = obs_all["source_file"].reset_index(drop=True).iloc[plot_idx].to_numpy()
for col in stage_cols[:4]:
    umap_df[col] = obs_all[col].reset_index(drop=True).iloc[plot_idx].astype(str).to_numpy()
umap_df.to_csv(OUT_DIR / "full_umap_streaming_plot_sample.csv.gz", index=False)

plt.figure(figsize=(7, 6))
sns.scatterplot(data=umap_df, x="UMAP1", y="UMAP2", hue="source_file", s=1, linewidth=0, alpha=0.35)
plt.legend(markerscale=6, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("Full-data streaming UMAP sample by source file")
savefig("full_umap_streaming_sample_by_file.png")

In [ ]:
candidate_color_cols = ["source_file"] + stage_cols[:4]
candidate_color_cols = [c for c in candidate_color_cols if c in obs_all.columns or c == "source_file"]

if "umap_df" in globals():
    plot_df = umap_df.copy()
    for col in stage_cols[:4]:
        if col in obs_all.columns and col not in plot_df.columns:
            aligned = obs_all[[col]].copy()
            aligned.index = aligned.index.astype(str)
            plot_df = plot_df.join(aligned, how="left")
    for col in [c for c in candidate_color_cols if c in plot_df.columns and c != "source_file"]:
        top = plot_df[col].astype(str).value_counts().head(20).index
        tmp = plot_df[plot_df[col].astype(str).isin(top)].copy()
        plt.figure(figsize=(7, 6))
        sns.scatterplot(data=tmp, x="UMAP1", y="UMAP2", hue=col, s=1, linewidth=0, alpha=0.35)
        plt.legend(markerscale=6, bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.title(f"Full-data UMAP by {col}")
        savefig(f"full_umap_by_{safe_name(col)}.png")

print("EDA complete.")